# Baseline Model — Bloomy Knowledge Tracing

Input: `data/processed/modeling_dataset.parquet`. Feature selection below comes directly from `notebooks/feature_relationships.ipynb`'s findings (keep/drop/redundancy) plus one addition agreed afterward: `skill_base_rate`, a leakage-safe per-skill baseline rate, replacing raw `skill_id` as the way to give the model skill-difficulty information (see Section 1).

Two models, in order:
- **Model 0 (floor)**: `p = prior_correct_rate`, no fitting — the number any real model has to beat.
- **Model 1**: logistic regression — chosen because it directly optimizes log loss (the project's primary metric, `docs/problem_statement.md` §5) and its coefficients are interpretable, matching the founder-legible-reasoning project goal.

Reasoning for these choices, and why not MLflow / not class-weighting / not a tree model yet, is written up in `docs/modeling.md`. Experiment results are logged to `results/metrics.json` (a notebook cell writes to it and reads it back) rather than a tracking server — this project has ~2 runs, not a sweep.

In [1]:
import json
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, log_loss
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

pd.set_option("display.max_columns", None)

df = pd.read_parquet("../data/processed/modeling_dataset.parquet")
train = df[df["split"] == "train"].copy()
test = df[df["split"] == "test"].copy()
print("full:", df.shape, "train:", train.shape, "test:", test.shape)

full: (417216, 17) train: (383613, 17) test: (33603, 17)


## 1. Feature preparation

Applying the feature_relationships.ipynb findings:
- **Drop** `prior_correct_count` (VIF 15.4 — mechanically `prior_correct_rate` x `n_prior_attempts`), `prior_hint_count_mean` (redundant with `prior_hint_used_rate`, keeping the rate), `prior_ms_first_response_mean` and `prior_overlap_time_mean` (near-zero/weak correlation, mutually redundant — dropping both).
- **Keep** `prior_correct_rate`, `n_prior_attempts`, `prior_hint_used_rate`, `prior_attempt_count_mean` (weak alone but low-VIF, not redundant), and same-row `original`/`answer_type`/`tutor_mode`.
- **New**: `skill_base_rate` — per-`skill_id` mean `target_correct`, computed **only** from train rows, then joined onto both train and test by `skill_id`. Test rows get the train-derived average for their skill, never their own or other test rows' outcomes. A `skill_id` in test but absent from train falls back to the overall train mean.

In [2]:
NUMERIC_FEATURES = ["prior_correct_rate", "n_prior_attempts", "prior_hint_used_rate", "prior_attempt_count_mean", "skill_base_rate"]
CATEGORICAL_FEATURES = ["original", "answer_type", "tutor_mode"]
FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES

skill_rate = train.groupby("skill_id")["target_correct"].mean()
global_rate = train["target_correct"].mean()

train["skill_base_rate"] = train["skill_id"].map(skill_rate)
test["skill_base_rate"] = test["skill_id"].map(skill_rate).fillna(global_rate)

unseen = set(test["skill_id"]) - set(skill_rate.index)
print(f"global train correct rate: {global_rate:.4f}")
print(f"skill_ids in test not present in train: {len(unseen)} (fallback to global rate)")
print("\nFinal feature set:", FEATURES)
train[FEATURES + ["target_correct"]].head()

global train correct rate: 0.6903
skill_ids in test not present in train: 2 (fallback to global rate)

Final feature set: ['prior_correct_rate', 'n_prior_attempts', 'prior_hint_used_rate', 'prior_attempt_count_mean', 'skill_base_rate', 'original', 'answer_type', 'tutor_mode']


,prior_correct_rate,n_prior_attempts,prior_hint_used_rate,prior_attempt_count_mean,skill_base_rate,original,answer_type,tutor_mode,target_correct
0,0.000000,1,1.000000,1.0,0.518584,1,algebra,tutor,1
1,0.500000,2,0.500000,1.0,0.518584,1,algebra,tutor,0
2,0.333333,3,0.666667,1.0,0.518584,1,algebra,tutor,0
3,0.250000,4,0.750000,1.0,0.518584,1,algebra,tutor,0
4,0.200000,5,0.800000,1.0,0.518584,1,algebra,tutor,0


In [3]:
# leakage spot-check: skill_base_rate must equal a TRAIN-only mean, unaffected by test rows
sample_skill = train["skill_id"].iloc[0]
manual = train.loc[train["skill_id"] == sample_skill, "target_correct"].mean()
looked_up = train.loc[train["skill_id"] == sample_skill, "skill_base_rate"].iloc[0]
print(f"skill_id={sample_skill}: manual train-only mean={manual:.6f}, skill_base_rate column={looked_up:.6f}, match={np.isclose(manual, looked_up)}")

skill_id=2: manual train-only mean=0.518584, skill_base_rate column=0.518584, match=True


## 2. Experiment logging

Flat JSON log instead of MLflow (see `docs/modeling.md` for why). `log_run` upserts by `name` — re-executing this notebook overwrites a run's own record instead of duplicating it, so `results/metrics.json` always reflects the latest state of each named experiment.

In [4]:
RESULTS_PATH = Path("../results/metrics.json")


def log_run(name, feature_list, params, metrics, notes=""):
    RESULTS_PATH.parent.mkdir(parents=True, exist_ok=True)
    runs = json.loads(RESULTS_PATH.read_text()) if RESULTS_PATH.exists() else []
    runs = [r for r in runs if r["name"] != name]
    record = {
        "name": name,
        "timestamp": datetime.now(timezone.utc).isoformat(),
        "features": feature_list,
        "params": params,
        **metrics,
        "notes": notes,
    }
    runs.append(record)
    RESULTS_PATH.write_text(json.dumps(runs, indent=2))
    return record


def load_runs():
    if not RESULTS_PATH.exists():
        return pd.DataFrame()
    return pd.DataFrame(json.loads(RESULTS_PATH.read_text()))


def evaluate(y_true, p):
    p = np.clip(p, 1e-15, 1 - 1e-15)
    ll = log_loss(y_true, p)
    pr_auc = average_precision_score(1 - y_true, 1 - p)  # PR-AUC on the minority "incorrect" class
    return ll, pr_auc

## 3. Model 0 — floor baseline (no fitting)

`p = prior_correct_rate` directly. `log_loss` is clipped to avoid `-inf` when the rate is exactly 0 or 1. PR-AUC is computed on the "incorrect" class (`1 - target_correct`, `1 - p`) since that's the minority/diagnostically-interesting class.

In [5]:
train_ll0, train_pr0 = evaluate(train["target_correct"], train["prior_correct_rate"])
test_ll0, test_pr0 = evaluate(test["target_correct"], test["prior_correct_rate"])
print(f"Model 0 (floor) - train: log_loss={train_ll0:.4f}  pr_auc={train_pr0:.4f}")
print(f"Model 0 (floor) - test:  log_loss={test_ll0:.4f}  pr_auc={test_pr0:.4f}")

log_run(
    name="floor_prior_correct_rate",
    feature_list=["prior_correct_rate"],
    params={},
    metrics={
        "train_logloss": train_ll0, "test_logloss": test_ll0,
        "train_pr_auc": train_pr0, "test_pr_auc": test_pr0,
    },
    notes="No fitting; p = prior_correct_rate directly. Floor any real model must beat.",
)

Model 0 (floor) - train: log_loss=2.2994  pr_auc=0.5805
Model 0 (floor) - test:  log_loss=1.8880  pr_auc=0.5254


{'name': 'floor_prior_correct_rate',
 'timestamp': '2026-08-21T14:03:54.646918+00:00',
 'features': ['prior_correct_rate'],
 'params': {},
 'train_logloss': 2.2993899027328912,
 'test_logloss': 1.8879740025880682,
 'train_pr_auc': 0.5805313955223447,
 'test_pr_auc': 0.525400979550817,
 'notes': 'No fitting; p = prior_correct_rate directly. Floor any real model must beat.'}

**Interpretation.** The floor's log loss is *bad* — 2.2994 (train) / 1.8880 (test), far worse than the "always predict 69%" trivial baseline would score (`-[0.6903*ln(0.6903) + 0.3097*ln(0.3097)] ≈ 0.61`). This isn't a bug: `prior_correct_rate` is exactly 0.0 or 1.0 for any (student, skill) pair whose single prior attempt went one way (see the `head()` above — row 0 has `prior_correct_rate=0.0` from just 1 prior attempt), and log loss punishes a confident-and-wrong prediction severely (clipped at `p=1e-15`, `-ln(1e-15) ≈ 34.5` per wrong row). This is the history-thinness finding from `feature_relationships.ipynb` §7 showing up concretely: treating a rate computed from 1 data point as a calibrated probability is exactly the failure mode that section warned about. PR-AUC (0.5805 train / 0.5254 test) is more forgiving since it only cares about ranking, not calibration — and it's not far off Model 1's, which makes sense: the *ordering* information in `prior_correct_rate` is real even when its literal value is a bad probability.

## 4. Model 1 — logistic regression

Default `sklearn` params (`C=1.0`, no `class_weight`). Deliberately **not** using `class_weight="balanced"` — that would distort fitted probabilities away from the true base rate to favor threshold-based metrics, which actively hurts log loss (a calibration-sensitive metric). See `docs/modeling.md` for the full reasoning; this is a refinement of `docs/data_quality_and_leakage.md` §7's more generic "class weighting preferred" guidance, made after fixing log loss as the primary metric.

In [6]:
preprocessor = ColumnTransformer([
    ("num", StandardScaler(), NUMERIC_FEATURES),
    ("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL_FEATURES),
])

model = Pipeline([
    ("prep", preprocessor),
    ("clf", LogisticRegression(max_iter=1000)),
])

model.fit(train[FEATURES], train["target_correct"])

train_proba = model.predict_proba(train[FEATURES])[:, 1]
test_proba = model.predict_proba(test[FEATURES])[:, 1]

train_ll1, train_pr1 = evaluate(train["target_correct"], train_proba)
test_ll1, test_pr1 = evaluate(test["target_correct"], test_proba)
print(f"Model 1 (logistic regression) - train: log_loss={train_ll1:.4f}  pr_auc={train_pr1:.4f}")
print(f"Model 1 (logistic regression) - test:  log_loss={test_ll1:.4f}  pr_auc={test_pr1:.4f}")

log_run(
    name="logistic_regression_v1",
    feature_list=FEATURES,
    params={"C": 1.0, "max_iter": 1000, "class_weight": None},
    metrics={
        "train_logloss": train_ll1, "test_logloss": test_ll1,
        "train_pr_auc": train_pr1, "test_pr_auc": test_pr1,
    },
    notes="Default sklearn params, no class weighting (see docs/modeling.md).",
)

Model 1 (logistic regression) - train: log_loss=0.5123  pr_auc=0.6053
Model 1 (logistic regression) - test:  log_loss=0.4171  pr_auc=0.5312


{'name': 'logistic_regression_v1',
 'timestamp': '2026-08-21T14:03:56.468249+00:00',
 'features': ['prior_correct_rate',
  'n_prior_attempts',
  'prior_hint_used_rate',
  'prior_attempt_count_mean',
  'skill_base_rate',
  'original',
  'answer_type',
  'tutor_mode'],
 'params': {'C': 1.0, 'max_iter': 1000, 'class_weight': None},
 'train_logloss': 0.5122984940403205,
 'test_logloss': 0.41710062243837837,
 'train_pr_auc': 0.605324143044477,
 'test_pr_auc': 0.5311884587822622,
 'notes': 'Default sklearn params, no class weighting (see docs/modeling.md).'}

In [7]:
feature_names = model.named_steps["prep"].get_feature_names_out()
coefs = pd.Series(model.named_steps["clf"].coef_[0], index=feature_names).sort_values()
print(coefs)

num__prior_hint_used_rate        -0.147620
cat__answer_type_open_response   -0.010054
cat__answer_type_fill_in_1       -0.001772
num__prior_attempt_count_mean     0.013910
cat__answer_type_algebra          0.048801
num__n_prior_attempts             0.072967
cat__original_1                   0.131741
cat__tutor_mode_test              0.135287
cat__answer_type_choose_1         0.201419
cat__answer_type_choose_n         0.214108
num__skill_base_rate              0.311464
cat__tutor_mode_tutor             0.317214
cat__original_0                   0.320760
num__prior_correct_rate           0.820428
dtype: float64


**Interpretation.** Numeric coefficients are on the standardized scale ("per 1 std-dev"). `prior_correct_rate` dominates (0.820) — consistent with it being the strongest raw correlate in `feature_relationships.ipynb`. `skill_base_rate` (0.311) is the second-largest numeric coefficient — direct confirmation that the new feature is pulling real weight, matching the Section 6 finding that motivated adding it. `prior_hint_used_rate` is the only *negative* numeric coefficient (-0.148), correctly signed (more historical hint reliance → lower P(correct)). `n_prior_attempts` and `prior_attempt_count_mean` are small but nonzero (0.073, 0.014) — consistent with their weak-but-not-redundant status from the VIF check.

Note on the categorical coefficients: `OneHotEncoder` here doesn't drop a reference level, so every category (including e.g. both `original_0` and `original_1`) gets its own coefficient rather than being read relative to a dropped baseline — L2 regularization keeps this well-defined despite the redundancy with the intercept. So a level's coefficient is its own additive pull, not "vs. some reference category"; don't over-read the categorical magnitudes here (e.g. `tutor_mode_test`, n=221 in train) as reliable effects without more data.

## 5. Comparison

In [8]:
runs = load_runs()
runs[["name", "train_logloss", "test_logloss", "train_pr_auc", "test_pr_auc"]]

,name,train_logloss,test_logloss,train_pr_auc,test_pr_auc
0,floor_prior_correct_rate,2.299390,1.887974,0.580531,0.525401
1,logistic_regression_v1,0.512298,0.417101,0.605324,0.531188


**Interpretation.** Logistic regression clearly beats the floor on the metric that matters (log loss): 0.5123 → down from 2.2994 on train, 0.4171 → down from 1.8880 on test — a ~4.5x reduction in both. The PR-AUC gap is much smaller (0.6053 vs 0.5805 train, 0.5312 vs 0.5254 test), reinforcing the Model 0 interpretation above: `prior_correct_rate` alone already ranks students reasonably well, but its raw value is a poor *calibrated* probability (too many exact 0s/1s from thin history), which is exactly what fitting a model — even a simple linear one — fixes by learning how much to trust that rate and blending in the other features.

One more pattern worth naming honestly: **test log loss is lower than train for both models**, which is not the usual "watch for overfitting" direction. This isn't overfitting — it's a property of the `split` construction (`scripts/build_modeling_dataset.py`): `test` is always the *last* attempt in a (student, skill) group, which by definition has the most prior history (highest `n_prior_attempts`) of any row in that group. Train mixes in every earlier, thinner-history attempt too (including every pair's first-ever training example at `n_prior_attempts=1`). So test rows are systematically easier to predict well — more history behind their features — not evidence the model generalizes better than it fits. Worth keeping in mind when comparing future models on this same split.

## Summary

- **Floor (`prior_correct_rate` as-is)**: log loss 2.2994 train / 1.8880 test, PR-AUC 0.5805 train / 0.5254 test. Bad log loss, driven by thin-history rows where the rate is exactly 0 or 1 — a direct, concrete instance of the history-thinness concern raised in `feature_relationships.ipynb` §7 and `docs/problem_statement.md`'s assumptions.
- **Logistic regression** (`prior_correct_rate`, `n_prior_attempts`, `prior_hint_used_rate`, `prior_attempt_count_mean`, `skill_base_rate`, `original`, `answer_type`, `tutor_mode`, default params, no class weighting): log loss 0.5123 train / 0.4171 test, PR-AUC 0.6053 train / 0.5312 test. ~4.5x log-loss improvement over the floor — fitting even a simple model to calibrate away from the floor's overconfident 0/1 rates is worth it. `prior_correct_rate` remains the dominant coefficient (0.820); `skill_base_rate`, the feature added specifically from the Section 6 skill-heterogeneity finding, is the second-largest (0.311), confirming it's carrying real signal rather than being redundant with `prior_correct_rate`.
- Both models are logged in `results/metrics.json` for comparison against future runs.

**Named next steps** (not attempted here):
- **Gradient-boosted trees** (LightGBM/XGBoost) as a complexity comparison — could use raw `skill_id` natively (no need for the `skill_base_rate` workaround) and may capture non-linear/interaction effects the decile plots didn't rule out but LR can't exploit.
- **Hyperparameter tuning** of the logistic regression (regularization strength `C`) — skipped here in favor of establishing the floor-vs-fit comparison first, per the plan.
- **A shrunk/smoothed `prior_correct_rate`** for thin-history rows (e.g. a Bayesian-average toward `skill_base_rate` weighted by `n_prior_attempts`) — directly targets the failure mode the floor model exposed, and could be tried as a feature engineering change before reaching for a more complex model.
- Re-evaluate whether `test_pr_auc` being noticeably lower than `train_pr_auc` for both models (unlike the log-loss direction) is worth investigating once a second model exists for comparison — one data point isn't enough to tell signal from split-specific noise.